In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("HotelBookings.csv")

In [3]:
df = pd.read_csv("HotelBookings.csv")

# --- Reference stats from TRAINING data only; frozen and reused on the test file in Part 3 ---
TOP_AGENTS = (df["agent"].value_counts(normalize=True, dropna=False)
                .loc[lambda s: s >= 0.01].index.dropna())
TOP_COUNTRIES = df["country"].replace("CN", "CHN").value_counts().nlargest(10).index
ADR_GROUP_MEANS = (df[df["adr"] != 0]
                   .groupby(["hotel", "arrival_date_year", "arrival_date_week_number"])["adr"].mean())
GLOBAL_ADR_MEAN = df.loc[df["adr"] != 0, "adr"].mean()   # fallback for unseen combos

EXTRA_LOG_COLS = ["adr", "previous_bookings_not_canceled", "booking_changes"]
USE_ROOM_CHANGED = True   # set False if the group judges assigned_room_type to be post-outcome leakage

def engineer(d):
    d = d.copy()
    # ADR: impute zeros (before arrival_date_week_number is dropped), then clip
    z = d["adr"] == 0
    if z.any():
        keys = list(zip(d.loc[z, "hotel"], d.loc[z, "arrival_date_year"], d.loc[z, "arrival_date_week_number"]))
        d.loc[z, "adr"] = [ADR_GROUP_MEANS.get(k, GLOBAL_ADR_MEAN) for k in keys]
    d["adr"] = d["adr"].clip(lower=0, upper=1000)

    # Group's new variables
    d["arrival_date"] = pd.to_datetime(
        d["arrival_date_year"].astype(str) + "-" + d["arrival_date_month"] + "-" + d["arrival_date_day_of_month"].astype(str),
        format="%Y-%B-%d")
    d["arrival_dow"] = d["arrival_date"].dt.dayofweek
    d["has_children"] = ((d["children"].fillna(0) + d["babies"]) > 0).astype(int)
    d["room_changed"] = (d["reserved_room_type"] != d["assigned_room_type"]).astype(int)
    prior_total = d["previous_cancellations"] + d["previous_bookings_not_canceled"]
    d["has_history"] = (prior_total > 0).astype(int)
    d["prior_cancel_rate"] = (d["previous_cancellations"] / prior_total.replace(0, np.nan)).fillna(0)
    d["has_agent"] = d["agent"].notna().astype(int)
    d["top_agent"] = np.where(d["agent"].isna(), "None",
                     np.where(d["agent"].isin(TOP_AGENTS), d["agent"].fillna(0).astype(int).astype(str), "Other"))
    d["has_company"] = d["company"].notna().astype(int)
    d["has_parking"] = (d["required_car_parking_spaces"] > 0).astype(int)
    d["was_waitlisted"] = (d["days_in_waiting_list"] > 0).astype(int)
    d["log_lead_time"] = np.log1p(d["lead_time"])

    # Conversions (no rows are dropped, so this can also run on the test file)
    d["meal"] = d["meal"].replace("Undefined", "SC")
    d["country"] = d["country"].replace("CN", "CHN")
    d["country_grp"] = d["country"].where(d["country"].isin(TOP_COUNTRIES), "OTHER")

    # Log-transform the remaining skewed columns for KNN distances
    for c in EXTRA_LOG_COLS:
        d[c] = np.log1p(d[c])

    drop = ["is_canceled", "agent", "company", "required_car_parking_spaces", "days_in_waiting_list",
            "country", "reserved_room_type", "assigned_room_type", "arrival_date",
            "arrival_date_week_number", "children", "babies", "previous_cancellations",
            "lead_time"]                                   # lead_time replaced by log_lead_time
    if not USE_ROOM_CHANGED:
        drop.append("room_changed")
    return d.drop(columns=[c for c in drop if c in d.columns])

CAT_COLS = ["hotel", "arrival_date_month", "meal", "market_segment", "distribution_channel",
            "deposit_type", "customer_type", "top_agent", "country_grp"]

X = engineer(df)
y = df["is_canceled"]
NUM_COLS = [c for c in X.columns if c not in CAT_COLS]
print(X.shape, "| categorical:", len(CAT_COLS), "| numeric:", len(NUM_COLS))
X.head()

(69710, 29) | categorical: 9 | numeric: 20


,hotel,arrival_date_year,arrival_date_month,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,meal,market_segment,distribution_channel,...,room_changed,has_history,prior_cancel_rate,has_agent,top_agent,has_company,has_parking,was_waitlisted,log_lead_time,country_grp
0,City Hotel,2017,March,31,1,2,2,SC,Online TA,TA/TO,...,0,0,0.0,1,9,0,1,0,4.465908,DEU
1,City Hotel,2016,March,26,2,5,2,BB,Online TA,TA/TO,...,0,0,0.0,1,9,0,0,0,1.791759,OTHER
2,City Hotel,2017,June,12,1,0,2,BB,Direct,Direct,...,0,0,0.0,1,14,0,0,0,3.044522,PRT
3,City Hotel,2015,November,11,0,3,2,BB,Online TA,TA/TO,...,1,0,0.0,1,9,0,0,0,2.197225,BRA
4,Resort Hotel,2017,March,10,1,2,1,BB,Online TA,TA/TO,...,0,0,0.0,1,240,0,0,0,5.616771,GBR


In [5]:
# Pre-processing & Importing
SEED = 888
np.random.seed(SEED)

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
                             average_precision_score, balanced_accuracy_score, matthews_corrcoef,
                             log_loss, brier_score_loss, precision_recall_curve, roc_auc_score)


In [6]:
# Splitting Train Test 
Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.2, stratify=y, random_state=888)

pre = ColumnTransformer([("num", StandardScaler(), NUM_COLS),
                         ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS)])
pipe = Pipeline([("pre", pre),
                 ("knn", KNeighborsClassifier(n_neighbors=25, p=1, weights="distance", n_jobs=-1))])

cv = StratifiedKFold(5, shuffle=True, random_state=888)

In [7]:
# Tuning (GridSearchCV)
grid = {"pre__transformer_weights": [{"num": 1, "cat": w} for w in (0.5, 0.75, 1.0, 1.5)]}
gs = GridSearchCV(pipe, grid, scoring="average_precision", cv=cv, n_jobs=1).fit(Xtr, ytr)
print(gs.best_params_, round(gs.best_score_, 4))

c:\Users\Justin San\anaconda3\envs\dsa312-new\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\Justin San\anaconda3\envs\dsa312-new\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\Justin San\anaconda3\envs\dsa312-new\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Justin San\anaconda3\envs\dsa312-new\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\Justin San\anaconda3\

{'pre__transformer_weights': {'num': 1, 'cat': 1.5}} 0.7453


In [8]:
make_cal = lambda: CalibratedClassifierCV(gs.best_estimator_, method="isotonic", cv=5)

oof = cross_val_predict(make_cal(), Xtr, ytr, cv=cv, method="predict_proba")[:, 1]
prec, rec, thr = precision_recall_curve(ytr, oof)
f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
best_thr = thr[f1s.argmax()]

cal = make_cal().fit(Xtr, ytr)
proba = cal.predict_proba(Xva)[:, 1]
pred = proba >= best_thr

print("thr %.3f | Acc %.4f | Prec %.4f | Rec %.4f | F1 %.4f" % (
    best_thr, accuracy_score(yva, pred), precision_score(yva, pred),
    recall_score(yva, pred), f1_score(yva, pred)))
print("PR-AUC %.4f | ROC-AUC %.4f | BalAcc %.4f | MCC %.4f | LogLoss %.4f | Brier %.4f" % (
    average_precision_score(yva, proba), roc_auc_score(yva, proba),
    balanced_accuracy_score(yva, pred), matthews_corrcoef(yva, pred),
    log_loss(yva, proba), brier_score_loss(yva, proba)))
print(confusion_matrix(yva, pred))

thr 0.339 | Acc 0.8084 | Prec 0.6221 | Rec 0.7583 | F1 0.6835
PR-AUC 0.7552 | ROC-AUC 0.8781 | BalAcc 0.7928 | MCC 0.5536 | LogLoss 0.3775 | Brier 0.1196
[[8387 1752]
 [ 919 2884]]
